<a href="https://colab.research.google.com/github/EiMonSan-Ellie/LLM_RAG-Project/blob/main/Text_Classification_and_Compliant_Response.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installating and Importing the Necessary Libraries

In [ ]:
!pip install numpy==1.26.4

Restart the session.

In [ ]:
!pip install pandas==2.1.4

**The Llama 2 Model**

In [ ]:
# Installation for GPU llama-cpp-python
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28  --force-reinstall --upgrade --no-cache-dir -q

In [ ]:
!pip install huggingface_hub

In [ ]:
!pip install -q datasets==2.16.1

In [ ]:
# Import the hf_hub_download function from the Hugging Face Hub library
from huggingface_hub import hf_hub_download

# Import the Llama class from the llama_cpp library
from llama_cpp import Llama

In [ ]:
model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename = "llama-2-13b-chat.Q5_K_M.gguf" # the model is in gguf format

In [ ]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

In [ ]:
lcpp_llm = Llama(
    model_path=model_path,
    n_threads=2,
    n_batch=512,
    n_gpu_layers=43, # Change this value based on your model and your GPU VRAM pool.
    n_ctx=4096,
)

# Problem Statement 1: Text Classification - Sentiment Analysis

**Loading the IMDb Reviews Dataset**

In [ ]:
import random
import pandas as pd
import numpy as np
np.float_ = np.float64

from datasets import load_dataset

In [ ]:
dataset = load_dataset("imdb")

In [ ]:
imdb_train_df = dataset['train'].to_pandas()
imdb_test_df = dataset['test'].to_pandas()

In [ ]:
(imdb_train_df.shape, imdb_test_df.shape)

Both the training and testing DataFrames have the same shape: 25,000 rows and 2 columns.

In [ ]:
positive_examples = imdb_train_df.loc[imdb_train_df.label == 1, :].sample(3)
negative_examples = imdb_train_df.loc[imdb_train_df.label == 0, :].sample(3)

**positive_examples** containing three random positive sentiment examples, and

**negative_examples** containing three random negative sentiment examples.

In [ ]:
positive_examples

In [ ]:
negative_examples

In [ ]:
#concatenating the subsets of positive and negative sentiment examples (positive_examples and negative_examples)
examples = pd.concat([positive_examples, negative_examples]).to_json(orient='records')

**Sentiment Classification**

- In this problem statement, we will be working with the IMDb dataset of movie reviews, along with the labeled sentiment for each review.

- We will utilize a pre-trained Large Language Model as a Text Classification engine to do Sentiment Analysis. This will be done simply by prompting the model to play this role and giving it clear instructions to output as prediction only the Positive / Negative sentiment label associated with each review.


In [ ]:
import json
import numpy as np
from collections import Counter
from tqdm import tqdm


In [ ]:
system_message = """[INST]<<SYS>>Classify the sentiment of movie reviews presented in the input as 'positive' or 'negative'.
Movie reviews will be delimited by triple backticks in the input.
Answer only 'positive' or 'negative'. Do not explain your answer.

Instructions:
1. Carefully read the text of the review and consider the overall sentiment of the review
2. Estimate the probability of the review being positive

To re-iterate, your answer should strictly only contain the label: positive or negative.

Some examples of expected output are provided below as guidance.<</SYS>>[/INST]
"""

- The main content of the message provides instructions to the user on a specific task, which is to classify the sentiment of movie reviews as either 'positive' or 'negative'.
- It mentions that movie reviews will be delimited (separated) by triple backticks in the input.
- It emphasizes that the user should provide only the label ('positive' or 'negative') as the answer and should not explain their answer.
- Instructions are given, including reading the text of the review and estimating the probability of the review being positive.
The message also provides examples of expected output for guidance.

**Prompt Template**

In [ ]:
prompt_template = """
[INST] ```{input_data}``` [/INST]
{output}
"""

In [ ]:
## Initialize an empty string to store few-shot examples
few_shot_examples = ''

**Few-shot Learning**

"Few-shot Learning" is an ML paradigm where a model is trained to make predictions or perform tasks with very limited examples or "shots" of training data.

In traditional Machine Learning, models often require large amounts of labeled data for training. However, in few-shot learning, the goal is to enable a model to generalize and make accurate predictions or decisions when it has access to only a small number of examples.

In [ ]:
## Iterate through each example in the JSON data which was created earlier
for example in json.loads(examples):
        # Extract the input data (text) from the example, excluding the 'label'

    example_input = {i:example[i] for i in example if i!='label'}
    # Determine the sentiment prediction based on the 'label' value
    if example['label'] == 0:
        example_prediction = 'negative'
    else:
        example_prediction = 'positive'

    # Concatenate the input data and the predicted sentiment
    # using a template and add it to the 'few_shot_examples' string

    few_shot_examples += prompt_template.format(
        input_data=example_input['text'],  ###input_data is used in the prompt_template
        output=example_prediction          ###outpu is used in the prompt_template
    )

In [ ]:
test_rows = json.loads(
    imdb_test_df.sample(100).to_json(orient='records')
)

The test_rows will contain a list of dictionaries, where each dictionary represents a single row (example) from the **IMDb testing dataset**. This allows to work **with a subset of the testing data, specifically 100** randomly sampled rows, in a structured format within your Python code.

**Making Predictions with the LLM**

In [ ]:
## Initialize empty lists to store model predictions and ground truth values.
model_predictions, ground_truths = [], []

In [ ]:
## Iterate through each row in the test data with a progress bar
for row in tqdm(test_rows):
      # Extract the input data (text) from the current row, excluding the 'label'
    test_input = {i:row[i] for i in row if i!='label'}

        # Construct a few-shot prompt by combining system message, few-shot examples, and test input
    few_shot_prompt = (
        system_message + few_shot_examples +
        prompt_template.format(
            input_data=test_input['text'],
            output=''
        )
    )

    try:
        # Use the model (lcpp_llm) to generate a response based on the few-shot prompt
        response = lcpp_llm(
            prompt=few_shot_prompt,
            max_tokens=2,
            temperature=0,
            top_p=0.95,
            repeat_penalty=1.2,
            top_k=50,
            stop=['INST'], # Dynamic stopping when such token is detected.
            echo=False # do not return the prompt
        )
        # Extract the model's prediction from the response

        prediction = response["choices"][0]["text"]

        # Append the model's prediction to the 'model_predictions' list, lowercased and stripped of whitespace
        model_predictions.append(prediction.strip().lower())

        # Determine the ground truth label based on the row's 'label' value and append it to 'ground_truths'
        if row['label'] == 0:
            ground_truths.append('negative')
        else:
            ground_truths.append('positive')
    except ValueError as e:
          # Handle any ValueErrors that may occur during the process and continue with the next row

        print(e)
        continue

- The code iterates through each row in the test_rows dataset, displaying a progress bar using tqdm for tracking progress.

- For each row, it extracts the input data (text) from the current row, excluding the 'label', and stores it in the test_input dictionary.

- It constructs a few-shot prompt by combining a system message, a few-shot example, and the current test input. This prompt is designed to be used as input for the model.

- Inside a try-except block, it uses the machine learning model (lcpp_llm) to generate a response based on the few-shot prompt. Several parameters, such as max_tokens, temperature, and stop, are set to control the model's behavior during response generation.

- It extracts the model's prediction from the response and appends it to the model_predictions list. The prediction is converted to lowercase and stripped of whitespace for consistency.

- It determines the ground truth label based on the 'label' value in the current row and appends it to the ground_truths list.

- In case of a ValueError during the process, it prints an error message and continues with the next row.

- After the loop completes, model_predictions will contain the model's predictions for the test data, and ground_truths will contain the true or ground truth labels. These lists can then be used for evaluating the model's performance, such as calculating accuracy, precision, recall, or other relevant metrics.

In [ ]:
Counter(model_predictions)

The model_predictions list as follows:

'**negative**': The model made 53 predictions with the label 'negative'.

'**positive**': The model made 47 predictions with the label 'positive'.

In [ ]:
Counter(ground_truths)

This Counter object counts the occurrences of labels in the ground_truths list, which likely contains the true or ground truth labels for the test data.

**'negative'**: There are 56 instances in the dataset with the label 'negative'.

**'positive'**: There are 44 instances in the dataset with the label 'positive'.

**Accuracy on the Test Set**

In [ ]:
ground_truths = np.array(ground_truths)
model_predictions = np.array(model_predictions)


#ground_truths contains the true labels (ground truth) for a set of examples.
#model_predictions contains the labels predicted by a machine learning model for the same set of examples.

In [ ]:
(ground_truths == model_predictions).mean()

An accuracy value of **0.95 (or 95%)** indicates that the model's predictions match the true labels for approximately 95% of the examples in the dataset. In other words, the model is correct in its classification for the vast majority of the examples - an example of the state-of-the-art performance that LLMs achieve in NLP tasks.

**Precision**

In [ ]:
TP = ((model_predictions == 'positive') & (ground_truths == 'positive')).sum()
FP = ((model_predictions == 'positive') & (ground_truths == 'negative')).sum()
precision = TP / (TP+FP)

In [ ]:
precision

High accuracy (0.95) indicates that the model is making correct predictions for most movie reviews.

The high precision (0.9149) remains a positive sign, showing that when the model predicts a review as **"positive," or "negative " it is usually correct.**

The model is effective in correctly classifying both positive and negative sentiment

# Problem Statement 2: Complaint Response Generation - Text Generation

In this problem statement, we will be providing a dataset of complaint messages written by customers of a bank.

We will utilize a pre-trained Large Language Model as a Text Generation model, to generate the appropriate response to these complaints. This will also be done by providing specific instructions in the prompt that guide the model on what to keep in mind while giving this response.


In [ ]:
dataset = load_dataset("AdiOO7/Bank_Complaints")

In [ ]:
system_message = """[INST]<<SYS>>As a spokesperson person of a particular bank, you are tasked to give a public response to a user's complaint presented as input.
Instructions:
1. Carefully observe the intensity and severity of the complaint received as input.
2. Choose a carefully worded public response. You need to reply to every complaint, however, responding with "Company chooses not to provide a public response" is also a valid response.
Some examples of appropriate response are provided below as guidance.<</SYS>>[/INST]
"""

In [ ]:
prompt_template = """
[INST] {input_example} [/INST]
{output_example}
"""

In [ ]:
few_shot_examples = ''

In [ ]:
for i in range(5):
    sample_document = dataset['train'][random.randint(0, 1829)]
    user_input_example = sample_document['Input']
    assistant_output_example = sample_document['Response']

    few_shot_examples += prompt_template.format(
        input_example=user_input_example,
        output_example=assistant_output_example
    )

In [ ]:
test_document = dataset['train'][random.randint(0, 1829)]
new_complaint = test_document['Input']

In [ ]:
few_shot_prompt = (
    system_message +
    few_shot_examples +
    prompt_template.format(
        input_example=new_complaint,
        output_example=''
    )
)

In [ ]:
response = lcpp_llm(
    prompt=few_shot_prompt,
    max_tokens=256,
    temperature=0,
    top_p=0.95,
    repeat_penalty=1.2,
    top_k=50,
    stop=['INST'], # Dynamic stopping when such token is detected.
    echo=False # do not return the prompt
)

In [ ]:
new_complaint

In [ ]:
print(response["choices"][0]["text"])